# Level Breakout

Level Breakout (Horizontal S/R) \
Built on the dedicated **`engine.level_detector`** — the stateful horizontal-level detector: resistance / support / pullback levels seeded at confirmed pivots and tracked forward until *invalidated* (absolute / percent / ATR tolerance). \
This is a **different, richer** level source than `fractal_breakout`, which derives its S/R from the lighter `indicators.detect_swing_*` fractal pivots (see `fractal_breakout.ipynb`). \
It is the level-detector member of the `level_*` family — room to grow `level_bounce` (fade at the level) and `level_retest` (break-and-retest) on the shared `engine.strategies.level_base.LevelStrategyBase`.

__How the Level-Breakout Algorithm Determines Entry/Exit:__
- Detects horizontal S/R via `engine.level_detector.detect_all_levels` (resistance + support; optional pullback).
- Entry is **geometry-driven** (ignores the detector's S/R labels): at each bar, active levels are split by their position vs the prior close.
- Long Entry: close decisively clears an *overhead* level (`prev_close < level`, `close > level + buffer·ATR`).
- Short Entry: close breaks an *underlying* level (`prev_close > level`, `close < level − buffer·ATR`).
- Stop: **structural**, anchored on the broken level (`level ∓ mult·ATR`), seeded at entry; exit preset `structural_rr2` takes profit at 2R, stop-first.
- Native flip: exit when the close crosses back through any active level.
- Look-ahead free: a level is only usable from its confirmation bar (`start_idx + pivot_window`) and only until its causal invalidation bar.

## Configuration: automatic

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import StrategyConfig, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = StrategyConfig()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic StrategyConfig().
STRATEGY_OVERRIDES = {}      # e.g. {"level_pivot_window": 4, "level_breakout_buffer_atr": 0.5}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# NOTE: a non-None policy here applies to EVERY strategy run below and overrides
# their differing per-strategy defaults — leave None to keep each one's own.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

## Level Breakout

It's a direction flip, not different entry logic:
- level_breakout has one entry signal: breakout (close crosses a level-detector S/R level).
- level_breakout and level_breakout_inv are two separate classes — ride the breakout vs fade it.
- The _inv is the same signal traded in the opposite direction.

In [ ]:
# Import level-breakout strategy (built on engine.level_detector)
from engine.strategies import LevelBreakoutStrategy

In [ ]:
# Backtest level-breakout strategy
strategy = LevelBreakoutStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Level-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Inverse Level Breakout

In [ ]:
# Import inverse level-breakout strategy (fade the breakout)
from engine.strategies import InverseLevelBreakoutStrategy

In [ ]:
# Backtest inverse level-breakout strategy
strategy = InverseLevelBreakoutStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# inverse level-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()